<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/Qwen3dot8_27B_topo_cbp_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/topogpt

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [ ]:
!nvidia-smi

Sat Nov 14 17:32:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## train

In [ ]:
import warnings
import os

# Suppress all python-level warnings and HF logging noise
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import shutil
import hashlib
import math

# =====================================================================
# 0. GOOGLE DRIVE & LOCAL SCRATCH PATH SETUP
# =====================================================================
drive_base = "/content/drive/MyDrive/topogpt"
drive_cache_dir = os.path.join(drive_base, "hf_cache_qwen_fineweb")
local_cache_dir = "/content/local_hf_cache_qwen_fineweb"

os.makedirs(drive_cache_dir, exist_ok=True)
os.makedirs(local_cache_dir, exist_ok=True)

local_tmp = "/content/tmp"
os.makedirs(local_tmp, exist_ok=True)
for var in ("TMPDIR", "TEMP", "TMP"):
    os.environ[var] = local_tmp

if os.path.exists(drive_cache_dir) and os.listdir(drive_cache_dir) and not os.listdir(local_cache_dir):
    print("Copying cached weights from Google Drive to local disk...")
    shutil.copytree(drive_cache_dir, local_cache_dir, dirs_exist_ok=True)

os.environ["HF_HOME"] = local_cache_dir
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from torch.optim import Optimizer
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from datasets import load_dataset
from safetensors.torch import save_file

# =====================================================================
# 1. GLOBAL SEED CONFIGURATION
# =====================================================================
SEED = 123
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 2. CONSTANTS & DATASET (FINEWEB - FIXED TO UPPERCASE)
# =====================================================================
BEDROCK_PRIMES = [2, 3, 5, 7, 11, 13]
EULER_ATTENUATION = 0.9785142874
SAFETY_CONSTANT = EULER_ATTENUATION
PRIME_ANCHORS = BEDROCK_PRIMES

print("Loading FineWeb dataset slice...")
fineweb_dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    name="sample-10BT",
    split="train[:500]",
    cache_dir=local_cache_dir
)

# =====================================================================
# 3. TOPO-CBP CUSTOM OPTIMIZER CLASS (TOPO-2026)
# =====================================================================
class TopoAdamW(Optimizer):
    """
    Topological Continual Backpropagation (Topo-CBP) Custom Optimizer.
    Natively houses the TOPO-2026 Topological Governor framework.
    """
    def __init__(self, params, lr=1e-5, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2, amsgrad=False, embedding_layer=None,
                 prime_anchors=None, lambda_euler=SAFETY_CONSTANT):
        if embedding_layer is None:
            raise ValueError("Topological Governor requires 'embedding_layer' to be explicitly passed.")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, amsgrad=amsgrad)
        super(TopoAdamW, self).__init__(params, defaults)

        self.embedding_layer = embedding_layer
        self.prime_anchors = prime_anchors or PRIME_ANCHORS
        self.lambda_euler = lambda_euler

        weight_data = self.embedding_layer.weight.data
        vocab_size = weight_data.shape[0]
        self.valid_anchors = [idx for idx in self.prime_anchors if idx < vocab_size]

        # Bedrock Mapping: Cache bit-perfect baseline state arrays in silicon memory
        self.baseline_invariant_tensor_map = {
            idx: weight_data[idx].detach().clone() for idx in self.valid_anchors
        }

        serialized = torch.cat([t.flatten().float() for t in self.baseline_invariant_tensor_map.values()]).cpu().numpy().tobytes()
        self.certified_sha256_digest = hashlib.sha256(serialized).hexdigest()
        print(f"[TopoAdamW] Baseline Bedrock Locked. Certified Digest: {self.certified_sha256_digest[:16]}")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # GOVERNOR PHASE I: Pre-Step Gradient Clamping
        if self.embedding_layer.weight.grad is not None:
            grad = self.embedding_layer.weight.grad
            grad.mul_(self.lambda_euler)
            for idx in self.valid_anchors:
                grad[idx].zero_()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    if group['amsgrad']:
                        state['max_exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                if group['amsgrad']:
                    max_exp_avg_sq = state['max_exp_avg_sq']
                beta1, beta2 = group['betas']

                state['step'] += 1

                p.data.mul_(1.0 - group['lr'] * group['weight_decay'])
                exp_avg.mul_(beta1).add_(p.grad, alpha=1.0 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(p.grad, p.grad, value=1.0 - beta2)

                if group['amsgrad']:
                    torch.max(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    denom = (max_exp_avg_sq.sqrt() / math.sqrt(1.0 - beta2 ** state['step'])).add_(group['eps'])
                else:
                    denom = (exp_avg_sq.sqrt() / math.sqrt(1.0 - beta2 ** state['step'])).add_(group['eps'])

                step_size = group['lr'] / (1.0 - beta1 ** state['step'])
                p.addcdiv_(exp_avg, denom, value=-step_size)

        # GOVERNOR PHASE II: Post-Step Bitwise Silicon Lock
        weight_data = self.embedding_layer.weight.data
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            weight_data[idx].copy_(original_tensor)

        return loss

    def audit_telemetry(self):
        weight_data = self.embedding_layer.weight.data
        max_drift = 0.0
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            current_tensor = weight_data[idx]
            drift = torch.max(torch.abs(current_tensor - original_tensor)).item()
            if drift > max_drift:
                max_drift = drift

        snapshot_tensors = [weight_data[idx] for idx in self.valid_anchors]
        serialized = torch.cat([t.flatten().float() for t in snapshot_tensors]).detach().cpu().numpy().tobytes()
        current_hash = hashlib.sha256(serialized).hexdigest()

        hash_matched = (current_hash == self.certified_sha256_digest)
        return max_drift, current_hash, hash_matched

# =====================================================================
# 4. LOAD MODEL (QWEN/QWEN3.8-27B, 4-BIT NF4 + BFLOAT16 HEADS)
# =====================================================================
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected.")
print(f"GPU: {torch.cuda.get_device_name(0)} | "
      f"free: {torch.cuda.mem_get_info(0)[0] / 1024**3:.1f} GB")

model_id = "Qwen/Qwen3.8-27B"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=local_cache_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

config = AutoConfig.from_pretrained(model_id, cache_dir=local_cache_dir, trust_remote_code=True)
if getattr(config, "rope_scaling", None) is not None:
    if isinstance(config.rope_scaling, dict):
        scaling_type = config.rope_scaling.get("type", None)
        if scaling_type == "default" or scaling_type is None:
            config.rope_scaling = None

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=local_cache_dir,
    trust_remote_code=True,
)
model.config.use_cache = False
print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.1f} GB")

# Freeze all backbone layers
for p in model.parameters():
    p.requires_grad_(False)

# Convert embed_tokens and lm_head to bfloat16 for direct gradient updates
TRAIN_KEYS = ("embed_tokens", "lm_head")
trainable = {}

for name, module in model.named_modules():
    if any(k == name.split(".")[-1] for k in TRAIN_KEYS):
        if hasattr(module, "weight"):
            if module.weight.dtype not in (torch.bfloat16, torch.float32):
                module.to(torch.bfloat16)
            module.weight.requires_grad_(True)
            trainable[f"{name}.weight"] = module.weight

n_train = sum(p.numel() for p in trainable.values())
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable (no LoRA): {list(trainable.keys())}")
print(f"trainable params: {n_train:,} || all params: {n_total:,} || "
      f"trainable%: {100 * n_train / n_total:.4f}")

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# =====================================================================
# 5. TOPO-CBP OPTIMIZER
# =====================================================================
embedding_layer = model.get_input_embeddings()

optimizer = TopoAdamW(
    list(trainable.values()),
    lr=5e-5,
    weight_decay=1e-2,
    embedding_layer=embedding_layer,
    prime_anchors=PRIME_ANCHORS,
    lambda_euler=SAFETY_CONSTANT,
)
baseline_hash = optimizer.certified_sha256_digest
print(f"Baseline hash (seed {SEED}): {baseline_hash}")

# =====================================================================
# 6. TOKENIZE FINEWEB DATA
# =====================================================================
tokenized = fineweb_dataset.map(
    lambda ex: tokenizer(ex["text"], truncation=True, max_length=1024),
    batched=True, remove_columns=fineweb_dataset.column_names,
)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# =====================================================================
# 7. HF TRAINER EXECUTION
# =====================================================================
class TopoTelemetryCallback(TrainerCallback):
    """Prints the Topo-CBP audit after every optimizer step."""
    def __init__(self, optimizer):
        self.optimizer = optimizer

    def on_step_end(self, args, state, control, **kwargs):
        d, _, matched = self.optimizer.audit_telemetry()
        print(f"step {state.global_step:3d}/{state.max_steps} | "
              f"drift {d:.10f} | hash match {matched}")

local_output_dir = "/content/qwen38-27b-topocbp-fineweb"
drive_output_dir = os.path.join(drive_base, "qwen38-27b-topocbp-fineweb")

num_epochs = 3
batch_size = 1
grad_accum = 4
total_steps = math.ceil(len(tokenized) / (batch_size * grad_accum)) * num_epochs
warmup_steps = max(1, int(0.10 * total_steps))

training_args = TrainingArguments(
    output_dir=local_output_dir,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=num_epochs,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=1,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED,
    report_to="none",
    disable_tqdm=True,
)

print(f"LR: {training_args.learning_rate} | Total Steps: {total_steps} | Warmup Steps: {warmup_steps}")

model._hf_peft_config_loaded = True
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=collator,
    optimizers=(optimizer, None),
    callbacks=[TopoTelemetryCallback(optimizer)],
)
model._hf_peft_config_loaded = False

print("Starting FineWeb training with Topo-CBP on Qwen3.8-27B...")
train_result = trainer.trainer.train() if hasattr(trainer, "trainer") else trainer.train()
loss_log = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]

# =====================================================================
# 8. POST-TRAINING CHECKS
# =====================================================================
final_drift, final_hash, hash_matched = optimizer.audit_telemetry()

print("\n" + "=" * 70)
print(f"Max Invariance Drift:                 {final_drift:.10f}")
print(f"Hash match (certified digest):        {hash_matched}")
print(f"Final hash:                           {final_hash}")
print("=" * 70)
print(f"Training Complete. Max Invariance Drift: {final_drift:.10f}")
print(f"INTEGRITY VERDICT: {'PASS' if hash_matched and final_drift == 0.0 else 'FAIL'}")

# =====================================================================
# 9. SAVE TRAINED WEIGHTS & SYNC TO GOOGLE DRIVE
# =====================================================================
os.makedirs(local_output_dir, exist_ok=True)

save_file(
    {n: p.detach().to(torch.bfloat16).cpu().contiguous() for n, p in trainable.items()},
    os.path.join(local_output_dir, "topo_cbp_trained_weights.safetensors"),
)
tokenizer.save_pretrained(local_output_dir)
with open(os.path.join(local_output_dir, "training_log.txt"), "w") as f:
    f.write(f"base_model: {model_id}\n")
    f.write(f"trained_tensors: {list(trainable.keys())}\n")
    f.write(f"baseline_hash: {baseline_hash}\n")
    f.write(f"final_hash: {final_hash}\n")
    f.write(f"hash_match: {hash_matched}\n")
    f.write(f"max_invariance_drift: {final_drift:.10f}\n")
    for st, l in loss_log:
        f.write(f"step {st}: {l:.6f}\n")

print("Copying outputs to Google Drive...")
if os.path.exists(drive_output_dir):
    shutil.rmtree(drive_output_dir)
shutil.copytree(local_output_dir, drive_output_dir)
print(f"Saved to {drive_output_dir}")

Loading FineWeb dataset slice...


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | free: 94.4 GB
Loading Qwen/Qwen3.8-27B...


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

Model memory footprint: 16.1 GB
Trainable (no LoRA): ['model.embed_tokens.weight', 'lm_head.weight']
trainable params: 2,542,796,800 || all params: 14,720,720,384 || trainable%: 17.2736
[TopoAdamW] Baseline Bedrock Locked. Certified Digest: c8ff8b2251e94c3a
Baseline hash (seed 123): c8ff8b2251e94c3a5a2b1d80713e3cd1a34bf5ac70813c20cf6cbaa952696593


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

LR: 5e-05 | Total Steps: 375 | Warmup Steps: 37
Starting FineWeb training with Topo-CBP on Qwen3.8-27B...
step   1/375 | drift 0.0000000000 | hash match True
{'loss': '2.778', 'grad_norm': '2.594', 'learning_rate': '0', 'epoch': '0.008'}
step   2/375 | drift 0.0000000000 | hash match True
{'loss': '2.367', 'grad_norm': '3.438', 'learning_rate': '1.351e-06', 'epoch': '0.016'}
step   3/375 | drift 0.0000000000 | hash match True
{'loss': '2.387', 'grad_norm': '2.75', 'learning_rate': '2.703e-06', 'epoch': '0.024'}
step   4/375 | drift 0.0000000000 | hash match True
{'loss': '2.022', 'grad_norm': '2.344', 'learning_rate': '4.054e-06', 'epoch': '0.032'}
step   5/375 | drift 0.0000000000 | hash match True
{'loss': '2.292', 'grad_norm': '2.359', 'learning_rate': '5.405e-06', 'epoch': '0.04'}
step   6/375 | drift 0.0000000000 | hash match True
{'loss': '2.45', 'grad_norm': '2.703', 'learning_rate': '6.757e-06', 'epoch': '0.048'}
step   7/375 | drift 0.0000000000 | hash match True
{'loss': '3.0

## evaluation-local0

In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from safetensors.torch import load_file

# =====================================================================
# PATH & DEVICE CONFIGURATION
# =====================================================================
model_id = "Qwen/Qwen3.8-27B"
checkpoint_dir = "/content/drive/MyDrive/topogpt/qwen38-27b-topocbp-fineweb"
weights_path = os.path.join(checkpoint_dir, "topo_cbp_trained_weights.safetensors")

print(f"Loading tokenizer from {checkpoint_dir}...")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model {model_id} in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# =====================================================================
# INJECT TOPO-CBP TRAINED WEIGHTS
# =====================================================================
print(f"Injecting Topo-CBP trained weights from {weights_path}...")
state_dict = load_file(weights_path)

for name, param in model.named_parameters():
    if name in state_dict:
        param.data.copy_(state_dict[name].to(param.device))
        print(f"Injected updated weights into: {name}")

model.eval()

# =====================================================================
# EVALUATION PROMPTS & GENERATION LOOP
# =====================================================================
math_prompts = [
    "Prove that the sum of the first n odd numbers is n^2 using mathematical induction.",
    "What is the closed-form solution for the Fibonacci sequence recurrence relation?",
    "Explain how analytic continuation extends the domain of the Riemann zeta function."
]

print("\n" + "=" * 50)
print("RUNNING MATHEMATICAL GENERATION EVALUATION")
print("=" * 50)

for prompt in math_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nPrompt: {prompt}\n")
    print(f"Response:\n{response}\n")
    print("-" * 50)

Loading tokenizer from /content/drive/MyDrive/topogpt/qwen38-27b-topocbp-fineweb...
Loading base model Qwen/Qwen3.8-27B in 4-bit NF4...


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Injecting Topo-CBP trained weights from /content/drive/MyDrive/topogpt/qwen38-27b-topocbp-fineweb/topo_cbp_trained_weights.safetensors...
Injected updated weights into: model.embed_tokens.weight
Injected updated weights into: lm_head.weight

RUNNING MATHEMATICAL GENERATION EVALUATION

Prompt: Prove that the sum of the first n odd numbers is n^2 using mathematical induction.

Response:
Prove that the sum of the first n odd numbers is n^2 using mathematical induction.

To prove this, we start with the base case: when \( n = 1 \), the sum of a single odd number (which is 1) equals \( 1^2 = 1^1 \). This establishes the initial condition. Next, assume it holds for some integer \( k \):
\[ S_k = 1 + 3 + \cdots + (2k-1) = k^2 \]
We need to show it holds for \( k+1 \):
S_{k+1} = S_k + (2(k+1)-1) = S_k +(2k+1) = (by assumption) k^2 + 2k + 1 = (k+1)^2
Thus, by the principle of mathematical induction, the statement holds for all natural numbers \( n \geq 1 \).What are the key steps in applying ma

## evaluation-local

In [ ]:
import torch
import os
import math
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from safetensors.torch import load_file

# =====================================================================
# PATH & DEVICE CONFIGURATION
# =====================================================================
model_id = "Qwen/Qwen3.8-27B"
checkpoint_dir = "/content/drive/MyDrive/topogpt/qwen38-27b-topocbp-fineweb"
weights_path = os.path.join(checkpoint_dir, "topo_cbp_trained_weights.safetensors")

print(f"Loading tokenizer from {checkpoint_dir}...")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model {model_id} in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# =====================================================================
# INJECT TOPO-CBP TRAINED WEIGHTS
# =====================================================================
print(f"Injecting Topo-CBP trained weights from {weights_path}...")
state_dict = load_file(weights_path)

for name, param in model.named_parameters():
    if name in state_dict:
        param.data.copy_(state_dict[name].to(param.device))
        print(f"Injected updated weights into: {name}")

model.eval()

# =====================================================================
# PERPLEXITY EVALUATION LOOP
# =====================================================================
print("\n" + "=" * 50)
print("COMPUTING VALIDATION PERPLEXITY")
print("=" * 50)

# Load a small validation slice from FineWeb
val_dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    name="sample-10BT",
    split="train[500:550]",
    cache_dir=checkpoint_dir
)

nlls = []
total_tokens = 0

with torch.no_grad():
    for example in val_dataset:
        encodings = tokenizer(example["text"], return_tensors="pt", truncation=True, max_length=1024)
        input_ids = encodings.input_ids.to(model.device)

        if input_ids.shape[1] <= 1:
            continue

        target_ids = input_ids.clone()
        outputs = model(input_ids, labels=target_ids)

        # Loss is averaged over tokens by default; multiply by sequence length to get total negative log-likelihood
        neg_log_likelihood = outputs.loss * (input_ids.shape[1] - 1)
        nlls.append(neg_log_likelihood.item())
        total_tokens += (input_ids.shape[1] - 1)

perplexities = [nll / total_tokens for nll in nlls]
mean_loss = sum(nlls) / total_tokens
perplexity = math.exp(mean_loss)

print(f"Validation Loss: {mean_loss:.4f}")
print(f"Validation Perplexity: {perplexity:.4f}")
print("=" * 50)

# =====================================================================
# MATHEMATICAL GENERATION EVALUATION
# =====================================================================
math_prompts = [
    "Prove that the sum of the first n odd numbers is n^2 using mathematical induction.",
    "What is the closed-form solution for the Fibonacci sequence recurrence relation?",
    "Explain how analytic continuation extends the domain of the Riemann zeta function."
]

print("\nRUNNING MATHEMATICAL GENERATION EVALUATION")
print("=" * 50)

for prompt in math_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nPrompt: {prompt}\n")
    print(f"Response:\n{response}\n")
    print("-" * 50)

Loading tokenizer from /content/drive/MyDrive/topogpt/qwen38-27b-topocbp-fineweb...
Loading base model Qwen/Qwen3.8-27B in 4-bit NF4...


config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

RuntimeError: Task error: File reconstruction error: IO Error: No space left on device (os error 28)

## mmlu & perplexity

In [ ]:
!pip install lm-eval -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import math
import json
import logging
from datetime import datetime
import torch
from datasets import load_dataset
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
import transformers.utils.import_utils as imp_utils
from warnings import filterwarnings

filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("lm_eval.models.huggingface").setLevel(logging.ERROR)
logging.getLogger("lm_eval.evaluator").setLevel(logging.ERROR)

TRAINED_PATH = "/content/drive/MyDrive/topogpt/qwen38-27b-topocbp-fineweb/topo_cbp_trained_weights.safetensors"
MODEL_ID = "Qwen/Qwen3.8-27B"
LOG_FILE = "/content/drive/MyDrive/topogpt/qwen38_evaluation_metrics.json"
MAX_LENGTH = 2048
STRIDE = 512
MMLU_BATCH_SIZE = 4  # lower to 2 or 1 if you hit OOM


# ---------------------------------------------------------------------------
# Compatibility Patches for DynamicCache
# ---------------------------------------------------------------------------
if not hasattr(DynamicCache, "from_legacy_cache"):
    @classmethod
    def _from_legacy_cache(cls, past_key_values):
        cache = cls()
        if past_key_values is not None:
            for k, v in past_key_values:
                cache.update(k, v)
        return cache
    DynamicCache.from_legacy_cache = _from_legacy_cache

if not hasattr(DynamicCache, "get_usable_length"):
    DynamicCache.get_usable_length = lambda self, new_seq_length=None, layer_idx=0: self.get_seq_length(layer_idx if isinstance(layer_idx, int) else 0)

if not hasattr(DynamicCache, "to_legacy_cache"):
    def _to_legacy_cache(self):
        return tuple((layer.keys, layer.values) for layer in self.layers)
    DynamicCache.to_legacy_cache = _to_legacy_cache
# ---------------------------------------------------------------------------


def fix_rotary_buffers(model):
    n = 0
    for _, mod in model.named_modules():
        if "RotaryEmbedding" in type(mod).__name__ and (hasattr(mod, "inv_freq") or hasattr(mod, "dim")):
            dev = mod.inv_freq.device if hasattr(mod, "inv_freq") else torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
            if dev.type == "meta":
                dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

            # Find the correct base/theta attribute name
            base_val = getattr(mod, "base", None) or getattr(mod, "rope_theta", None) or getattr(mod, "theta", 10000.0)
            dim_val = getattr(mod, "dim", None) or getattr(mod, "head_dim", 128)

            inv_freq = 1.0 / (
                base_val ** (torch.arange(0, dim_val, 2, dtype=torch.float32, device=dev) / dim_val)
            )
            mod.register_buffer("inv_freq", inv_freq, persistent=False)
            if hasattr(mod, "_set_cos_sin_cache"):
                seq = getattr(mod, "max_seq_len_cached", None) or getattr(mod, "max_position_embeddings", 4096)
                mod._set_cos_sin_cache(seq_len=seq, device=dev, dtype=torch.get_default_dtype())
            n += 1
    print(f"Recomputed rotary buffers in {n} modules")



@torch.inference_mode()
def evaluate_perplexity(model, tokenizer, dataset_name="Salesforce/wikitext",
                        dataset_config="wikitext-2-raw-v1", split="test"):
    print(f"Loading dataset {dataset_name} ({dataset_config} - {split})...")
    data = load_dataset(dataset_name, dataset_config, split=split)

    text_data = "\n\n".join(data["text"])
    encoded = tokenizer(text_data, return_tensors="pt")
    input_ids = encoded.input_ids.to(model.device)

    seq_len = input_ids.size(1)
    print(f"Total evaluation tokens: {seq_len:,}")

    total_nll = 0.0
    total_valid_tokens = 0
    loss_fct = torch.nn.CrossEntropyLoss(reduction="none")

    prev_end = 0
    for begin_loc in range(0, seq_len, STRIDE):
        end_loc = min(begin_loc + MAX_LENGTH, seq_len)
        trg_len = end_loc - prev_end

        input_chunk = input_ids[:, begin_loc:end_loc]
        target_chunk = input_chunk.clone()
        target_chunk[:, :-trg_len] = -100

        outputs = model(input_ids=input_chunk, use_cache=False)
        logits = outputs.logits.float()

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = target_chunk[:, 1:].contiguous()

        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        valid_mask = shift_labels.view(-1) != -100
        n_valid = int(valid_mask.sum().item())

        if n_valid > 0:
            total_nll += loss[valid_mask].sum().item()
            total_valid_tokens += n_valid

        prev_end = end_loc
        if end_loc >= seq_len:
            break

    ppl = math.exp(total_nll / total_valid_tokens) if total_valid_tokens > 0 else float("inf")
    print(f"Scored tokens: {total_valid_tokens:,}")
    return ppl


def save_metrics_log(metrics_dict):
    logs = []
    if os.path.exists(LOG_FILE):
        try:
            with open(LOG_FILE, "r") as f:
                logs = json.load(f)
        except json.JSONDecodeError:
            logs = []

    if logs and logs[-1].get("timestamp") == metrics_dict.get("timestamp"):
        logs[-1] = metrics_dict
    else:
        logs.append(metrics_dict)

    os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)
    with open(LOG_FILE, "w") as f:
        json.dump(logs, f, indent=4, default=str)
    print(f"Metrics successfully logged to {LOG_FILE}")


def run_lm_eval_harness(model, tokenizer):
    try:
        import lm_eval
        from lm_eval.models.huggingface import HFLM
    except ImportError:
        print("\nlm-evaluation-harness not installed. Install via `pip install lm-eval` to run MMLU.")
        return None

    try:
        print("\nInitializing LM Evaluation Harness for MMLU...")
        lm_obj = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=MMLU_BATCH_SIZE)
        return lm_eval.simple_evaluate(
            model=lm_obj,
            tasks=["mmlu"],
            num_fewshot=5,
        )
    except Exception as e:
        print(f"\nMMLU evaluation failed: {type(e).__name__}: {e}")
        return None


def main():
    if not os.path.exists(TRAINED_PATH):
        print(f"Trained checkpoint not found at: {TRAINED_PATH}")
        return

    if not hasattr(imp_utils, "is_torch_fx_available"):
        imp_utils.is_torch_fx_available = lambda: False

    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    if isinstance(getattr(config, "rope_scaling", None), dict):
        if config.rope_scaling.get("type", None) in ["default", None]:
            config.rope_scaling = None
    config.use_cache = False

    print("Loading base model in 4-bit NF4...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        config=config,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    model.config.use_cache = False
    fix_rotary_buffers(model)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    print("Applying Topo-CBP boundary weights...")
    state = load_file(TRAINED_PATH, device="cpu")
    targets = {
        "model.embed_tokens.weight": model.get_input_embeddings().weight,
        "lm_head.weight": model.get_output_embeddings().weight,
    }
    with torch.no_grad():
        for key, param in targets.items():
            if key not in state:
                raise KeyError(f"{key} missing from checkpoint: {list(state.keys())}")
            param.copy_(state[key].to(device=param.device, dtype=param.dtype))

    # --- Perplexity ---
    ppl = evaluate_perplexity(model, tokenizer)

    metrics = {
        "timestamp": datetime.now().isoformat(),
        "model_id": MODEL_ID,
        "checkpoint_path": TRAINED_PATH,
        "wikitext2_perplexity": ppl,
    }
    save_metrics_log(metrics)

    print("\n==================================================")
    print(" EVALUATION RESULTS (WikiText-2 Test Split)")
    print("==================================================")
    print(f" Perplexity (PPL): {ppl:.4f}")
    print("==================================================")

    # --- MMLU ---
    mmlu_results = run_lm_eval_harness(model, tokenizer)
    if mmlu_results:
        res = mmlu_results.get("results", {})
        mmlu_summary = res.get("mmlu", res)
        print("\n==================================================")
        print(" MMLU EVALUATION RESULTS (5-shot)")
        print("==================================================")
        print(json.dumps(mmlu_summary, indent=2, default=str))
        print("==================================================")
        metrics["mmlu"] = mmlu_summary
        save_metrics_log(metrics)


if __name__ == "__main__":
    main()

Loading base model in 4-bit NF4...


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Recomputed rotary buffers in 1 modules


tokenizer_config.json:   0%|          | 0.00/9.98k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (526 outputs left out)

Generating validation split:   0%|          | 0/19 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 56168/56168 [1:13:50<00:00, 12.68it/s]



 MMLU EVALUATION RESULTS (5-shot)
{
  "alias": "mmlu",
  "name": "mmlu",
  "sample_len": 14042,
  "acc,none": 0.6385130323315767,
  "acc_stderr,none": 0.003839473170924848,
  "sample_count": {
    "acc,none": 14042
  }
}
Metrics successfully logged to /content/drive/MyDrive/topogpt/qwen38_evaluation_metrics.json


## gguf

In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file

MODEL_ID = "Qwen/Qwen3.8-27B"
CHECKPOINT_DIR = "/content/drive/MyDrive/topogpt/qwen38-27b-topocbp-fineweb"
WEIGHTS_PATH = os.path.join(CHECKPOINT_DIR, "topo_cbp_trained_weights.safetensors")
MERGED_OUTPUT_DIR = "/content/qwen38-27b-topocbp-bf16-merged"

print("Loading base model in full BFloat16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu",  # Load on CPU RAM to avoid VRAM limits during merge
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR, trust_remote_code=True)

print("Injecting Topo-CBP trained weights...")
state_dict = load_file(WEIGHTS_PATH)
targets = {
    "model.embed_tokens.weight": model.get_input_embeddings().weight,
    "lm_head.weight": model.get_output_embeddings().weight,
}

with torch.no_grad():
    for key, param in targets.items():
        if key in state_dict:
            param.copy_(state_dict[key].to(device=param.device, dtype=param.dtype))
            print(f"Merged updated weights into: {key}")

print(f"Saving merged BF16 model to {MERGED_OUTPUT_DIR}...")
os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)
model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)
print("Merge complete! Ready for GGUF conversion.")

Loading base model in full BFloat16...


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Injecting Topo-CBP trained weights...
Merged updated weights into: model.embed_tokens.weight
Merged updated weights into: lm_head.weight
Saving merged BF16 model to /content/qwen38-27b-topocbp-bf16-merged...


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merge complete! Ready for GGUF conversion.


In [ ]:
!ls /content/drive/MyDrive/topogpt

In [ ]:
!ls /content/

'=0.46.1'   drive   qwen38-27b-topocbp-bf16-merged   sample_data


In [ ]:
!python /content/drive/MyDrive/topogpt/llama.cpp/convert_hf_to_gguf.py /content/qwen38-27b-topocbp-bf16-merged \
    --outfile /content/drive/MyDrive/topogpt/qwen3.8-27b-topocbp-bf16.gguf \
    --outtype f16

In [ ]:
!/content/drive/MyDrive/topogpt/tmpdev/build_quant/bin/llama-quantize /content/drive/MyDrive/topogpt/qwen3.8-27b-topocbp-bf16.gguf \
                           /content/drive/MyDrive/GGUF_Models/qwen3.8-27b-topocbp-Q8_0.gguf \
                           Q8_0

In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/*.gguf

-rw------- 1 root root  27G Nov 14 20:17 /content/drive/MyDrive/GGUF_Models/qwen3.8-27b-topocbp-Q8_0.gguf
-rw------- 1 root root  11G Nov 12 06:55 /content/drive/MyDrive/GGUF_Models/deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw------- 1 root root  70G Nov  9 06:01 /content/drive/MyDrive/GGUF_Models/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
-rw------- 1 root root  32G Nov  9 05:57 /content/drive/MyDrive/GGUF_Models/sarvam-30b-topo-governed-q8_0.gguf
-rw------- 1 root root  16G Nov  9 04:40 /content/drive/MyDrive/GGUF_Models/muse-glimmer-30b-card-faithful-Q4_K_M.gguf
-rw------- 1 root root  15G Nov  9 04:36 /content/drive/MyDrive/GGUF_Models/gpt-oss-20b-topo-governed-Q4_K_M.gguf
-rw------- 1 root root  15G Nov  9 04:34 /content/drive/MyDrive/GGUF_Models/certified-scorpion-topo-qwen-bf16.gguf
-rw------- 1 root root  40G Nov  9 04:31 /content/drive/MyDrive/GGUF_Models/llama-3.1-70b-topo-governed-q4_k_m.gguf
-rw------- 1 root root 5.0G Nov  9 04:23 /content/drive/MyDrive/GGUF_Models/

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

In [ ]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 81152 MiB):
  Device 0: NVIDIA A100-SXM4-80GB, compute capability 8.0, VMM: yes, VRAM: 81152 MiB


GPU Offload Supported: True


In [ ]:
import os
import struct

def inspect_gguf_metadata(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    file_size_gb = os.path.getsize(file_path) / (1024 ** 3)
    print(f"Inspecting GGUF File: {file_path}")
    print(f"Total File Size: {file_size_gb:.2f} GB\n")

    with open(file_path, "rb") as f:
        # Read GGUF magic bytes
        magic = f.read(4)
        if magic != b"GGUF":
            print("Error: Invalid GGUF magic header!")
            return

        version = struct.unpack("<I", f.read(4))[0]
        tensor_count = struct.unpack("<Q", f.read(8))[0]
        kv_count = struct.unpack("<Q", f.read(8))[0]

        print("--- GGUF Header Summary ---")
        print(f"Magic: {magic.decode('ascii', errors='ignore')}")
        print(f"GGUF Version: {version}")
        print(f"Total Tensors: {tensor_count:,}")
        print(f"Metadata KV Count: {kv_count:,}\n")

        # Helper to read GGUF strings
        def read_gguf_string():
            length = struct.unpack("<Q", f.read(8))[0]
            return f.read(length).decode("utf-8", errors="ignore")

        print("--- Key Metadata Parameters ---")
        # Iterate through metadata KV pairs
        for i in range(kv_count):
            try:
                key = read_gguf_string()
                value_type = struct.unpack("<I", f.read(4))[0]

                # Parse value based on GGUF type enum
                if value_type == 8:  # String
                    val = read_gguf_string()
                    print(f"  {key}: {val}")
                elif value_type in (4, 5, 6):  # u32, i32, f32
                    val = struct.unpack("<f" if value_type == 6 else "<i", f.read(4))[0]
                    print(f"  {key}: {val}")
                elif value_type == 7:  # bool
                    val = struct.unpack("<?", f.read(1))[0]
                    print(f"  {key}: {val}")
                elif value_type == 10:  # i64 / u64
                    val = struct.unpack("<q", f.read(8))[0]
                    print(f"  {key}: {val}")
                else:
                    # Skip complex/array types for clean summary printing
                    break
            except Exception as e:
                print(f"  [Reached end of readable metadata header at index {i}]")
                break

if __name__ == "__main__":
    model_path = "/content/drive/MyDrive/GGUF_Models/qwen3.8-27b-topocbp-Q8_0.gguf"
    inspect_gguf_metadata(model_path)

Inspecting GGUF File: /content/drive/MyDrive/GGUF_Models/qwen3.8-27b-topocbp-Q8_0.gguf
Total File Size: 26.63 GB

--- GGUF Header Summary ---
Magic: GGUF
GGUF Version: 3
Total Tensors: 851
Metadata KV Count: 44

--- Key Metadata Parameters ---
  general.architecture: qwen35
  general.type: model
  general.sampling.top_k: 20
  general.sampling.top_p: 0.949999988079071
  general.sampling.temp: 1.0
  general.name: Qwen38 27b Topocbp Bf16 Merged
  general.finetune: topocbp-merged
  general.basename: qwen38
  general.size_label: 27B
  qwen35.block_count: 65
  qwen35.context_length: 262144
  qwen35.embedding_length: 5120
  qwen35.feed_forward_length: 17408
  qwen35.attention.head_count: 24
  qwen35.attention.head_count_kv: 4


In [ ]:
import os
import struct
from collections import Counter

def inspect_gguf_tensors(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    # Map GGUF GGML type integers to readable names
    ggml_type_map = {
        0: "F32",
        1: "F16",
        2: "Q4_0",
        3: "Q4_1",
        7: "Q8_0",
        8: "Q5_0",
        9: "Q5_1",
        10: "Q2_K",
        12: "Q4_K",
        14: "Q6_K",
        29: "BF16",
    }

    with open(file_path, "rb") as f:
        # Read header
        magic = f.read(4)
        if magic != b"GGUF":
            print("Error: Invalid GGUF magic header!")
            return

        version = struct.unpack("<I", f.read(4))[0]
        tensor_count = struct.unpack("<Q", f.read(8))[0]
        kv_count = struct.unpack("<Q", f.read(8))[0]

        # Helper to read GGUF strings
        def read_string():
            length = struct.unpack("<Q", f.read(8))[0]
            return f.read(length).decode("utf-8", errors="ignore")

        # Skip metadata KV pairs to reach tensor info
        for _ in range(kv_count):
            read_string()  # key
            value_type = struct.unpack("<I", f.read(4))[0]

            # Skip values based on GGUF type
            if value_type == 8:  # String
                read_string()
            elif value_type in (4, 5, 6):  # u32, i32, f32
                f.read(4)
            elif value_type == 7:  # bool
                f.read(1)
            elif value_type in (1, 2, 3, 10, 11, 12):  # i8, u8, i16, i64, u64, f64
                f.read(8)
            elif value_type == 9:  # Array
                arr_type = struct.unpack("<I", f.read(4))[0]
                arr_len = struct.unpack("<Q", f.read(8))[0]
                # Fast skip array data if possible or parse basic types
                if arr_type == 8:  # array of strings
                    for _ in range(arr_len):
                        read_string()
                else:
                    # Approximate skip based on type size
                    type_sizes = {4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}
                    f.read(arr_len * type_sizes.get(arr_type, 4))

        print(f"Scanning {tensor_count} tensors in GGUF file...\n")
        tensor_types = Counter()
        tensor_details = []

        for i in range(tensor_count):
            name = read_string()
            n_dimensions = struct.unpack("<I", f.read(4))[0]
            dimensions = [struct.unpack("<Q", f.read(8))[0] for _ in range(n_dimensions)]
            tensor_type_val = struct.unpack("<I", f.read(4))[0]
            offset = struct.unpack("<Q", f.read(8))[0]

            type_name = ggml_type_map.get(tensor_type_val, f"UNKNOWN_{tensor_type_val}")
            tensor_types[type_name] += 1
            tensor_details.append((name, dimensions, type_name))

        print("=" * 50)
        print(" TENSOR DATA TYPE BREAKDOWN")
        print("=" * 50)
        for t_type, count in tensor_types.most_common():
            print(f"  - {t_type}: {count} tensors")
        print("=" * 50)

        print("\nFirst 10 Tensors Sample:")
        for name, dims, t_type in tensor_details[:10]:
            print(f"  [{t_type}] {name} -> Shape: {dims}")

if __name__ == "__main__":
    model_path = "/content/drive/MyDrive/GGUF_Models/qwen3.8-27b-topocbp-Q8_0.gguf"
    inspect_gguf_tensors(model_path)

Scanning 851 tensors in GGUF file...

 TENSOR DATA TYPE BREAKDOWN
  - Q5_0: 498 tensors
  - F32: 353 tensors

First 10 Tensors Sample:
  [Q5_0] output.weight -> Shape: [5120, 248320]
  [F32] output_norm.weight -> Shape: [5120]
  [Q5_0] token_embd.weight -> Shape: [5120, 248320]
  [Q5_0] blk.0.attn_gate.weight -> Shape: [5120, 6144]
  [F32] blk.0.attn_norm.weight -> Shape: [5120]
  [Q5_0] blk.0.attn_qkv.weight -> Shape: [5120, 10240]
  [Q5_0] blk.0.ffn_down.weight -> Shape: [17408, 5120]
  [Q5_0] blk.0.ffn_gate.weight -> Shape: [5120, 17408]
  [Q5_0] blk.0.ffn_up.weight -> Shape: [5120, 17408]
  [F32] blk.0.post_attention_norm.weight -> Shape: [5120]


In [ ]:
!apt-get update && !apt-get install -y git
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --force-reinstall --no-cache-dir git+https://github.com/abetlen/llama-cpp-python.git --verbose

In [ ]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

In [ ]:
import os
from llama_cpp import Llama

model_path = "/content/drive/MyDrive/GGUF_Models/qwen3.8-27b-topocbp-Q8_0.gguf"

print(f"Loading Topo-CBP governed Qwen 3.8-27B (Q8_0)...")

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,  # Full GPU offload
    verbose=False
)

prompt = (
    "<|im_start|>system\n"
    "You are a rigorous mathematical AI assistant governed by topological invariant locking.\n"
    "<|im_end|>\n"
    "<|im_start|>user\n"
    "Prove that the sum of the first n odd numbers is n^2 using mathematical induction.\n"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
)

output = llm(
    prompt,
    max_tokens=512,
    temperature=0.7,
    repeat_penalty=1.15,
    stop=["<|im_end|>"]
)

print("\n--- Generation Output ---")
print(output["choices"][0]["text"].strip())

Loading Topo-CBP governed Qwen 3.8-27B (Q8_0)...

--- Generation Output ---
<think>
The user asks me to prove that the sum of the first n odd numbers is n² using mathematical induction. This is a classic proof by induction. Let me structure this carefully.

The statement: 1 + 3 + 5 + ... + (2n-1) = n² for all positive integers n.

I need to:
1. State the proposition clearly
2. Base case: verify for n = 1
3. Inductive hypothesis: assume true for some k
4. Inductive step: prove it holds for k+1
5. Conclude by induction

Let me write this up rigorously and completely.
</think>

# Proof: The Sum of the First *n* Odd Numbers Equals *n*²

## Statement to Prove

For every positive integer $n$:

$$\sum_{i=1}^{n}(2i - 1) \;=\; n^2$$

That is,

$$1 + 3 + 5 + \cdots + (2n-1) = n^2.$$

We proceed by **mathematical induction** on $n$.

---

## Base Case ($n = 1$)

The sum of the first odd number is simply:

$$\sum_{i=1}^{1}(2i-1) = 2(1)-1 = 1.$$

And the right-hand side gives:

$$1^2 = 1. \qquad \c

In [ ]:
!nvidia-smi

Sat Nov 14 21:17:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             57W /  400W |   27434MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## hf

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

# Retrieve your token from Colab Secrets and authenticate
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

user_or_name = 'frankmorales2020'
repo_name = 'qwen3.8-27b-topocbp-gguf'
repo_id = f"{user_or_name}/{repo_name}"

local_file_path = "/content/drive/MyDrive/GGUF_Models/qwen3.8-27b-topocbp-Q8_0.gguf"
path_in_repo = "qwen3.8-27b-topocbp-Q8_0.gguf"

print(f"Initializing upload to Hugging Face repository: {repo_id}...")

api = HfApi()

# Ensure the repository exists (creates it if it doesn't already exist)
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

# Upload the GGUF model file
api.upload_file(
    path_or_fileobj=local_file_path,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model",
)

print(f"Upload complete! Model is successfully published at: https://huggingface.co/{repo_id}")

Initializing upload to Hugging Face repository: frankmorales2020/qwen3.8-27b-topocbp-gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3.8-27b-topocbp-Q8_0.gguf:   0%|          |  569kB / 28.6GB            

Validating                    :           |   0%            

Upload complete! Model is successfully published at: https://huggingface.co/frankmorales2020/qwen3.8-27b-topocbp-gguf


## inference-hf

In [ ]:
!apt-get update && !apt-get install -y git
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --force-reinstall --no-cache-dir git+https://github.com/abetlen/llama-cpp-python.git --verbose

In [ ]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 81152 MiB):
  Device 0: NVIDIA A100-SXM4-80GB, compute capability 8.0, VMM: yes, VRAM: 81152 MiB


GPU Offload Supported: True


In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Authenticate with your Hugging Face secret token
HF_TOKEN = userdata.get('HF_TOKEN')

repo_id = "frankmorales2020/qwen3.8-27b-topocbp-gguf"
filename = "qwen3.8-27b-topocbp-Q8_0.gguf"

print(f"Downloading {filename} from Hugging Face repository {repo_id}...")
model_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    token=HF_TOKEN
)
print(f"Model downloaded to local cache path: {model_path}")

print("Initializing Llama instance with full GPU offload...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False
)

prompt = (
    "<|im_start|>system\n"
    "You are a rigorous mathematical AI assistant governed by topological invariant locking.\n"
    "<|im_end|>\n"
    "<|im_start|>user\n"
    "Prove that the sum of the first n odd numbers is n^2 using mathematical induction.\n"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
)

print("Running inference test...")
output = llm(
    prompt,
    max_tokens=512,
    temperature=0.7,
    repeat_penalty=1.15,
    stop=["<|im_end|>"]
)

print("\n--- Generation Output ---")
print(output["choices"][0]["text"].strip())

Model downloaded to local cache path: /root/.cache/huggingface/hub/models--frankmorales2020--qwen3.8-27b-topocbp-gguf/snapshots/711b0768b4e53464eba898dc7a7b3b0645c709c6/qwen3.8-27b-topocbp-Q8_0.gguf
Initializing Llama instance with full GPU offload...
Running inference test...

--- Generation Output ---
<think>
The user asks me to prove that the sum of the first n odd numbers is n² using mathematical induction. This is a classic proof by induction. Let me structure this carefully.

The statement: 1 + 3 + 5 + ... + (2n-1) = n² for all positive integers n.

I need to:
1. State the proposition clearly
2. Base case: verify for n = 1
3. Inductive hypothesis: assume true for some k
4. Inductive step: prove it holds for k+1
5. Conclude by induction

Let me write this up rigorously and completely.
</think>

# Proof: The Sum of the First *n* Odd Numbers Equals *n*²

## Statement to Prove

For every positive integer $n$:

$$\sum_{i=1}^{n}(2i - 1) \;=\; n^2$$

That is,

$$1 + 3 + 5 + \cdots + (2n

In [ ]:
import time
from llama_cpp import Llama

# Assumes model_path is already set from your previous download
# model_path = "/root/.cache/huggingface/hub/models--frankmorales2020--qwen3.8-27b-topocbp-gguf/snapshots/.../qwen3.8-27b-topocbp-Q8_0.gguf"

print("Initializing benchmark instance...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False
)

prompt = (
    "<|im_start|>system\n"
    "You are a rigorous mathematical AI assistant governed by topological invariant locking.\n"
    "<|im_end|>\n"
    "<|im_start|>user\n"
    "Write a comprehensive explanation of the Riemann Hypothesis and its connection to prime number distribution.\n"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
)

print("\n--- Running Benchmark Generation ---")
start_time = time.time()
output = llm(
    prompt,
    max_tokens=512,
    temperature=0.7,
    repeat_penalty=1.15,
    stop=["<|im_end|>"]
)
end_time = time.time()

# Extract token metrics provided by llama-cpp-python usage stats
usage = output.get("usage", {})
completion_tokens = usage.get("completion_tokens", 0)
prompt_tokens = usage.get("prompt_tokens", 0)
total_time = end_time - start_time

print(f"\n--- Benchmark Results ---")
print(f"Prompt Tokens: {prompt_tokens}")
print(f"Completion Tokens: {completion_tokens}")
print(f"Total Time: {total_time:.2f} seconds")
if completion_tokens > 0:
    print(f"Generation Throughput: {completion_tokens / total_time:.2f} tokens/sec")

print("\nGenerated Text Preview:")
print(output["choices"][0]["text"].strip())

Initializing benchmark instance...

--- Running Benchmark Generation ---

--- Benchmark Results ---
Prompt Tokens: 49
Completion Tokens: 512
Total Time: 14.90 seconds
Generation Throughput: 34.37 tokens/sec

Generated Text Preview:
<think>
The user is asking for a comprehensive explanation of the Riemann Hypothesis and its connection to prime number distribution. This is a well-established mathematical topic, so I need to provide an accurate, thorough, and pedagogically structured response. Let me organize my thoughts:

1. Start with motivation - why primes matter
2. The Prime Number Theorem as background
3. Riemann's 1859 paper and the zeta function
4. Analytic continuation of ζ(s)
5. The functional equation
6. Definition of critical strip, trivial vs non-trivial zeros
7. Statement of the hypothesis precisely
8. Connection to prime distribution - the explicit formulas
9. What would be proven if RH is true (error term in PNT)
10. Current status and significance

Let me write this as a 

In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/*.gguf

In [ ]:
import os
import struct
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/GGUF_Models")

def read_gguf_header(filepath):
    try:
        with open(filepath, "rb") as f:
            magic = f.read(4)
            if magic != b"GGUF":
                return {"error": "Invalid magic bytes"}
            version = struct.unpack("<I", f.read(4))[0]
            tensor_count = struct.unpack("<Q", f.read(8))[0]
            kv_count = struct.unpack("<Q", f.read(8))[0]
            return {
                "version": version,
                "tensor_count": tensor_count,
                "kv_count": kv_count
            }
    except Exception as e:
        return {"error": str(e)}

print(f"{'Model Filename':<52} | {'Size (GB)':<10} | {'GGUF Version':<12} | {'Status'}")
print("-" * 95)

total_size = 0
for file_path in sorted(model_dir.glob("*.gguf")):
    file_size = file_path.stat().st_size
    size_gb = file_size / (1024**3)
    total_size += file_size

    meta = read_gguf_header(file_path)
    if "error" not in meta:
        status = "Verified"
        version_str = f"v{meta['version']}"
    else:
        status = meta["error"]
        version_str = "N/A"

    print(f"{file_path.name:<52} | {size_gb:>8.2f}GB | {version_str:<12} | {status}")

print("-" * 95)
print(f"Total Portfolio Storage: {total_size / (1024**3):.2f} GB across {len(list(model_dir.glob('*.gguf')))} models.")

Model Filename                                       | Size (GB)  | GGUF Version | Status
-----------------------------------------------------------------------------------------------
certified-scorpion-topo-qwen-bf16.gguf               |    14.19GB | v3           | Verified
deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf |    69.83GB | v3           | Verified
deepseek_moe_16b_topocbp_q4_k_m.gguf                 |    10.11GB | v3           | Verified
gpt-oss-20b-topo-governed-Q4_K_M.gguf                |    14.72GB | v3           | Verified
llama-3.1-70b-topo-governed-q4_k_m.gguf              |    39.60GB | v3           | Verified
mixtral_8x22b_q4_k_m.gguf                            |    79.71GB | v3           | Verified
muse-glimmer-30b-card-faithful-Q4_K_M.gguf           |    15.77GB | v3           | Verified
nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf          |     4.97GB | v3           | Verified
qwen3.8-27b-topocbp-Q8_0.gguf                        |    26.63GB | v3       